# 04 · Violence vs non-violence classification on RLVS (video level)

A cleaner, smaller task than 14-class UCF-Crime: **4,000 clips (2,000 per class), two classes** (Real Life Violence Situations, Kaggle).

Method: 16 frames per clip → frozen CLIP ViT-B/16 embeddings → (a) logistic regression and (b) a small temporal transformer trained for **60 epochs**.

How we keep the result honest:
- **Split by group, not by clip.** Many RLVS clips are cut from the same source video. Near-duplicate clips are grouped (CLIP similarity) so they never sit on both sides of a split.
- **The test clips are used once**, after all choices (regularisation, best epoch, which model) were made on validation clips only.
- We also report a plain random-clip split, which can leak near-duplicates, so you can see how much that inflates the number.
- A bootstrap 95% interval is reported because the test set is only a few hundred clips.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. **Secrets:** `KAGGLE_USERNAME`, `KAGGLE_KEY` (enable notebook access).

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q kagglehub

In [ ]:
import os, json, time
from pathlib import Path
import numpy as np, cv2, torch
from google.colab import drive, userdata

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/crime_scene_runs/rlvs"
os.makedirs(OUT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
import kagglehub
DATA_DIR = kagglehub.dataset_download("mohamedmustafa/real-life-violence-situations-dataset")
print("data:", DATA_DIR, "| device:", device)

## 1. Classifier code (same file as `training/violence_classifier.py`)

In [ ]:
%%writefile violence_classifier.py
"""Video-level violence classifier on frozen CLIP frame features (RLVS / RWF-2000 style data).

Pipeline: 16 frames per clip -> CLIP image embeddings (computed in the notebook) -> this module:
  * groups near-duplicate clips so they cannot land on both sides of a split (RLVS cuts many clips from the same source),
  * splits by group into train / validation / test,
  * fits a logistic-regression baseline and a small temporal transformer head (trained for 60 epochs),
  * evaluates once on the held-out test clips, with a bootstrap confidence interval.
Model selection uses validation data only; the test set is never used to choose anything.
"""
import numpy as np


def clip_video_features(frame_feats):
    """(N, T, D) frame embeddings -> (N, 2D) clip features: mean and max over time."""
    return np.concatenate([frame_feats.mean(1), frame_feats.max(1)], axis=1)


def choose_groups(frame_feats, thresholds=(0.97, 0.95, 0.93, 0.91, 0.89, 0.87), max_frac=0.05):
    """Group clips whose mean embeddings are near-duplicates (cosine >= threshold).

    Picks the lowest threshold whose largest group is still <= max_frac of all clips, so no giant group forms.
    Returns (group_ids, threshold, sizes_of_groups).
    """
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import connected_components

    v = frame_feats.mean(1)
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    sim = v @ v.T
    np.fill_diagonal(sim, 0.0)
    best = None
    for th in thresholds:
        n, g = connected_components(csr_matrix(sim >= th), directed=False)
        sizes = np.bincount(g)
        if sizes.max() <= max_frac * len(g):
            best = (g, th, sizes)
    if best is None:  # even the strictest threshold made a big group: fall back to one clip per group
        g = np.arange(len(v))
        best = (g, None, np.ones(len(v), dtype=int))
    return best


def split_by_group(groups, test_size=0.2, val_size=0.15, seed=0):
    """Indices for train / val / test with no group shared between sets."""
    from sklearn.model_selection import GroupShuffleSplit

    idx = np.arange(len(groups))
    outer = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval, test = next(outer.split(idx, groups=groups))
    inner = GroupShuffleSplit(n_splits=1, test_size=val_size / (1 - test_size), random_state=seed + 1)
    tr, va = next(inner.split(trainval, groups=groups[trainval]))
    train, val = trainval[tr], trainval[va]
    assert not (set(groups[train]) & set(groups[test])) and not (set(groups[val]) & set(groups[test]))
    assert not (set(groups[train]) & set(groups[val]))
    return train, val, test


def fit_logreg(xtr, ytr, xva, yva, cs=(0.01, 0.1, 1.0, 10.0)):
    """Standardise, then pick C on the validation set. Returns (predict_proba_fn, best_c, val_acc)."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    sc = StandardScaler().fit(xtr)
    best = None
    for c in cs:
        m = LogisticRegression(C=c, max_iter=2000).fit(sc.transform(xtr), ytr)
        acc = float((m.predict(sc.transform(xva)) == yva).mean())
        if best is None or acc > best[2]:
            best = (m, c, acc)
    m, c, acc = best
    return (lambda x: m.predict_proba(sc.transform(x))[:, 1]), c, acc


def train_head(ftr, ytr, fva, yva, epochs=60, lr=1e-3, wd=1e-2, batch=64, seed=0, device="cpu"):
    """Small temporal transformer over frame embeddings. Returns (predict_proba_fn, history, best_epoch)."""
    import torch
    import torch.nn as nn

    torch.manual_seed(seed)
    n, t, d = ftr.shape

    class Head(nn.Module):
        def __init__(self):
            super().__init__()
            self.norm = nn.LayerNorm(d)
            self.proj = nn.Linear(d, 256)
            self.pos = nn.Parameter(torch.zeros(1, t, 256))
            layer = nn.TransformerEncoderLayer(256, 4, 512, dropout=0.3, batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(layer, num_layers=1)
            self.drop = nn.Dropout(0.3)
            self.out = nn.Linear(256, 2)

        def forward(self, x):
            h = self.proj(self.norm(x)) + self.pos
            h = self.enc(h).mean(1)
            return self.out(self.drop(h))

    model = Head().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    steps = epochs * int(np.ceil(n / batch))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    xtr = torch.tensor(ftr, dtype=torch.float32, device=device)
    ytr_t = torch.tensor(ytr, dtype=torch.long, device=device)
    xva = torch.tensor(fva, dtype=torch.float32, device=device)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    hist, best_acc, best_state, best_epoch = [], -1.0, None, 0
    g = torch.Generator().manual_seed(seed)
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n, generator=g)
        tot = 0.0
        for i in range(0, n, batch):
            b = perm[i:i + batch].to(device)
            x = xtr[b]
            x = x + 0.02 * torch.randn_like(x)                                  # feature noise
            keep = (torch.rand(x.shape[0], t, 1, device=device) > 0.15).float()  # frame dropout
            x = x * keep
            opt.zero_grad()
            loss = loss_fn(model(x), ytr_t[b])
            loss.backward()
            opt.step()
            sched.step()
            tot += loss.item() * len(b)
        model.eval()
        with torch.no_grad():
            pva = model(xva).argmax(1).cpu().numpy()
        acc = float((pva == yva).mean())
        hist.append({"epoch": ep, "train_loss": tot / n, "val_acc": acc})
        if acc > best_acc:
            best_acc, best_epoch = acc, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()

    def proba(f):
        with torch.no_grad():
            return torch.softmax(model(torch.tensor(f, dtype=torch.float32, device=device)), 1)[:, 1].cpu().numpy()

    return proba, hist, best_epoch


def evaluate(y, prob, threshold=0.5, n_boot=1000, seed=0):
    """Clip-level metrics plus a bootstrap 95% interval for accuracy."""
    from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

    pred = (prob >= threshold).astype(int)
    rng = np.random.default_rng(seed)
    accs = [accuracy_score(y[i], pred[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(n_boot))]
    return {
        "n_clips": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "accuracy_ci95": [float(np.percentile(accs, 2.5)), float(np.percentile(accs, 97.5))],
        "macro_f1": float(f1_score(y, pred, average="macro")),
        "violence_precision": float(precision_score(y, pred, zero_division=0)),
        "violence_recall": float(recall_score(y, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y, prob)) if len(set(y)) > 1 else None,
        "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
    }

## 2. Find the clips and their labels

In [ ]:
EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
vids = sorted(p for p in Path(DATA_DIR).rglob("*") if p.suffix.lower() in EXTS)

def label_of(p):
    name = p.parent.name.lower().replace("_", "").replace("-", "").replace(" ", "")
    if name.startswith("non"):
        return 0
    if "violence" in name:
        return 1
    return None

labels = [label_of(p) for p in vids]
keep = [i for i, l in enumerate(labels) if l is not None]
vids = [vids[i] for i in keep]
y = np.array([labels[i] for i in keep])
print(len(vids), "clips | violence:", int(y.sum()), "| non-violence:", int((1 - y).sum()))
print("example:", vids[0])
assert len(vids) > 100 and y.sum() > 0 and (1 - y).sum() > 0, "unexpected folder layout: inspect DATA_DIR and adjust label_of()"

## 3. Frame sampling and CLIP embeddings (cached on Drive)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from transformers import CLIPModel

N_FRAMES = 16
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch16", torch_dtype=torch.float16).to(device).eval()
MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


def read_frames(path, n=N_FRAMES, size=224):
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    frames = []
    if total > 0:
        for i in np.linspace(0, total - 1, n).astype(int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
            ok, f = cap.read()
            if ok:
                frames.append(f)
    if not frames:  # unknown length: read everything
        cap.release()
        cap = cv2.VideoCapture(str(path))
        allf = []
        while True:
            ok, f = cap.read()
            if not ok:
                break
            allf.append(f)
        if allf:
            frames = [allf[i] for i in np.linspace(0, len(allf) - 1, n).astype(int)]
    cap.release()
    if not frames:
        return None
    while len(frames) < n:
        frames.append(frames[-1])
    out = []
    for f in frames[:n]:
        h, w = f.shape[:2]
        s = size / min(h, w)
        f = cv2.resize(f, (max(size, round(w * s)), max(size, round(h * s))), interpolation=cv2.INTER_AREA)
        h, w = f.shape[:2]
        y0, x0 = (h - size) // 2, (w - size) // 2
        out.append(cv2.cvtColor(f[y0:y0 + size, x0:x0 + size], cv2.COLOR_BGR2RGB))
    return np.stack(out)


@torch.no_grad()
def embed(frames):
    x = torch.from_numpy(frames).to(device).permute(0, 3, 1, 2).float() / 255
    f = clip.get_image_features(pixel_values=((x - MEAN) / STD).half())
    if not torch.is_tensor(f):
        f = getattr(f, "pooler_output", f[0])
    return torch.nn.functional.normalize(f.float(), dim=-1).cpu().numpy()


CACHE = Path(OUT) / "rlvs_clip_feats.npz"
if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    feats, y, paths = z["feats"], z["y"], list(z["paths"])
    print("loaded cached features:", feats.shape)
else:
    rows, ok_idx, t0 = [], [], time.time()
    with ThreadPoolExecutor(4) as ex:
        for start in range(0, len(vids), 64):
            chunk = list(range(start, min(start + 64, len(vids))))
            for i, fr in zip(chunk, ex.map(read_frames, [vids[j] for j in chunk])):
                if fr is None:
                    continue
                rows.append(embed(fr))
                ok_idx.append(i)
            print(f"{chunk[-1] + 1}/{len(vids)} clips, {time.time() - t0:.0f}s", flush=True)
    feats = np.stack(rows).astype(np.float32)
    y = y[ok_idx]
    paths = [str(vids[i]) for i in ok_idx]
    np.savez_compressed(CACHE, feats=feats, y=y, paths=np.array(paths))
    print("features:", feats.shape, "| clips dropped (unreadable):", len(vids) - len(ok_idx))

## 4. Group near-duplicate clips and split

In [ ]:
import violence_classifier as vc

groups, th, sizes = vc.choose_groups(feats)
print(f"{len(set(groups))} groups (similarity threshold {th}); largest group {sizes.max()} clips; singleton groups {(sizes == 1).sum()}")
tr, va, te = vc.split_by_group(groups, test_size=0.2, val_size=0.15, seed=0)
print("train / val / test clips:", len(tr), len(va), len(te), "| violence share in test:", round(float(y[te].mean()), 3))

## 5. Train (60 epochs for the temporal head) and evaluate once on the test clips

In [ ]:
X = vc.clip_video_features(feats)
lr_proba, best_c, lr_val = vc.fit_logreg(X[tr], y[tr], X[va], y[va])
head_proba, hist, best_ep = vc.train_head(feats[tr], y[tr], feats[va], y[va], epochs=60, device=device)
head_val = max(h["val_acc"] for h in hist)
print(f"validation accuracy | logistic regression (C={best_c}): {lr_val:.3f} | temporal head (best epoch {best_ep}/60): {head_val:.3f}")

chosen = "temporal_head" if head_val >= lr_val else "logistic_regression"   # chosen on validation only
results = {
    "dataset": "RLVS (Kaggle: mohamedmustafa/real-life-violence-situations-dataset)",
    "n_clips": int(len(y)), "frames_per_clip": N_FRAMES, "features": "CLIP ViT-B/16, frozen",
    "split": "grouped by near-duplicate similarity", "group_threshold": th, "n_groups": int(len(set(groups))),
    "train_val_test_clips": [int(len(tr)), int(len(va)), int(len(te))],
    "chosen_on_validation": chosen,
    "logistic_regression": vc.evaluate(y[te], lr_proba(X[te])),
    "temporal_head": vc.evaluate(y[te], head_proba(feats[te])),
}

# For comparison only: a plain random clip split, which can leak near-duplicate clips between train and test
perm = np.random.default_rng(0).permutation(len(y))
te2, va2, tr2 = perm[:len(te)], perm[len(te):len(te) + len(va)], perm[len(te) + len(va):]
lr2, _, _ = vc.fit_logreg(X[tr2], y[tr2], X[va2], y[va2])
results["random_clip_split_logreg_for_comparison"] = vc.evaluate(y[te2], lr2(X[te2]))

for name in ("logistic_regression", "temporal_head", "random_clip_split_logreg_for_comparison"):
    r = results[name]
    print(f"{name:42s} acc {r['accuracy']:.3f}  95% CI [{r['accuracy_ci95'][0]:.3f}, {r['accuracy_ci95'][1]:.3f}]  macro-F1 {r['macro_f1']:.3f}  AUC {r['roc_auc']:.3f}")
print("chosen on validation:", chosen)
Path(OUT, "results.json").write_text(json.dumps(results, indent=2))

## 6. Curves and confusion matrix

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot([h["epoch"] for h in hist], [h["train_loss"] for h in hist]); ax[0].set_title("temporal head: train loss"); ax[0].set_xlabel("epoch")
ax[1].plot([h["epoch"] for h in hist], [h["val_acc"] for h in hist]); ax[1].axvline(best_ep, ls=":", c="gray")
ax[1].set_title(f"validation accuracy (best epoch {best_ep})"); ax[1].set_xlabel("epoch")
cm = np.array(results[chosen]["confusion_matrix"])
ax[2].imshow(cm, cmap="Blues"); ax[2].set_title(f"test confusion matrix ({chosen})")
ax[2].set_xticks([0, 1], ["non-violence", "violence"]); ax[2].set_yticks([0, 1], ["non-violence", "violence"])
for i in range(2):
    for j in range(2):
        ax[2].text(j, i, int(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
plt.tight_layout(); plt.savefig(Path(OUT) / "rlvs_curves.png", dpi=130); plt.show()
print(json.dumps(results[chosen], indent=2))

## What to report
Use the **chosen-on-validation** model's test numbers with their confidence interval, and mention the random-split number only as a comparison. RLVS includes some staged and movie fights, so this result does not show that the model works on real CCTV; a cross-dataset test on RWF-2000 would.